# Figures Notebook
This notebook was used to produce the figures and statistics in the publicly available paper.

Warning: Certain parts of this notebook *may* be outdated and require slight changes to run properly.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import math
sys.path.append('../')

import torch
from omegaconf import OmegaConf
from matplotlib import pyplot as plt

from src.models.nets import NETSModule
from src.models.ctds import CTDSModule
from src.simulation.dynamics import ForwardProcessSampleable
from src.utils.misc import get_device
from src.utils.plotting import plot_proposal, mt_plot_proposal, timeseries_plot_density, timeseries_plot_contours, mt_timeseries_plot_density, mt_timeseries_plot_contours

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Using device: {device}')

In [2]:
# Load models
save_dir = '../pretrained_checkpoints/learned'

# Learned paths
baseline_path = os.path.join(save_dir, 'baseline.ckpt')

overdamped_path = os.path.join(save_dir, 'nets_overdamped.ckpt')
overdamped_jarzynski_path = os.path.join(save_dir, 'nets_overdamped_jarzynski.ckpt')

inertial_path = os.path.join(save_dir, 'nets_inertial.ckpt')
inertial_jarzynski_path = os.path.join(save_dir, 'nets_inertial_jarzynski.ckpt')

ctds_path = os.path.join(save_dir, 'ctds.ckpt')
ctds_jarzynski_path = os.path.join(save_dir, 'ctds_jarzynski.ckpt')

### Models Figure

In [ ]:
from matplotlib import pyplot as plt

from src.models.nets import NETSModule
from src.models.ctds import CTDSModule
from src.simulation.dynamics import ForwardProcessSampleable
from src.utils.plotting import scatter_sampleable, plot_density, plot_contours
from src.systems.distributions import GMM

model_ts = torch.linspace(0, 1, 100).to(device)

# Reference
reference_pinn = NETSModule.load_from_checkpoint(baseline_path).move_to_device(device)
reference_model_sampleable = ForwardProcessSampleable(
    forward_process=reference_pinn.build_proposal("ode").dynamics,
    ts=model_ts
).to(device)

# Overdamped NETS
overdamped_jarzynski_pinn = NETSModule.load_from_checkpoint(overdamped_jarzynski_path).move_to_device(device)
overdamped_model_sampleable = ForwardProcessSampleable(
    forward_process=overdamped_jarzynski_pinn.build_proposal("ode").dynamics,
    ts=model_ts
).to(device)


# CTDS
ctds_pinn = CTDSModule.load_from_checkpoint(ctds_jarzynski_path)
ctds_model_sampleable = ForwardProcessSampleable(
    forward_process=ctds_pinn.build_proposal("ode").dynamics,
    ts=model_ts
).to(device)

# Create target
target = GMM.FAB_GMM(cov_scale=0.25).to(device)

# Create graphs
fig, axes = plt.subplots(1, 3, figsize=(20, 10))

# Plot density and contours on all subplots
for ax in axes.flatten():
    ax.set_xlim(-50, 50)
    ax.set_ylim(-50, 50)
    plot_density(target.log_density, ax=ax, scale=50, alpha=0.5, vmin=-40, zorder=0, device=device)
    plot_contours(target.log_density, ax=ax, scale=50, alpha=0.5, legend=False, levels=50, zorder=0, device=device)

scatter_args = {
    'num_samples': 1000,
    'marker': 'o',
    's': 10,
    'color': 'black',
    'alpha': 0.5,
    'zorder': 1,
}

# Reference model
ax = axes[1]
ax.set_title("Baseline")
scatter_sampleable(
    reference_model_sampleable,
    ax=ax,
    **scatter_args,
)


# Overdamped proposal
ax = axes[0]
ax.set_title("Overdamped NETS")
scatter_sampleable(
    overdamped_model_sampleable,
    ax=ax,
    **scatter_args,
)

# CTDS model
ax = axes[2]
ax.set_title("CTDS")
scatter_sampleable(
    ctds_model_sampleable,
    ax=ax,
    **scatter_args,
)

plt.show()

### Comparison of Temperature Distribution of Proposal

In [ ]:
from hydra import compose, initialize
from matplotlib import pyplot as plt
import seaborn as sns

def get_bs(dynamics, converter, nts: int = 500, num_samples: int = 2500):
    """
    Sample from the CTDS dynamics.
    """
    ts = torch.linspace(0, 1, nts).to(device).view(1, -1, 1).expand(num_samples, -1, -1)
    xz_pxz, _ = dynamics.sample(
        ts=ts, num_samples=num_samples, use_tqdm = True
    )
    z = xz_pxz[:, 2]
    b = converter.xi_to_beta(z)
    return b

# Grab proposal from uninitialized model
os.environ["RUN_CONFIG"] = "fab_tempered" 
with initialize(version_base=None, config_path="../config"):
    cfg = compose(config_name="ctds_config")

untrained_ctds = CTDSModule(cfg=cfg.ctds_config).move_to_device(device)
untrained_bs = get_bs(untrained_ctds.proposal.dynamics, untrained_ctds.converter).cpu().numpy()

# Grab proposal from initialized model
trained_ctds = CTDSModule.load_from_checkpoint(ctds_jarzynski_path).move_to_device(device)
trained_bs = get_bs(trained_ctds.proposal.dynamics, trained_ctds.converter).cpu().numpy()

In [ ]:
plt.figure(figsize=(8, 4), dpi=500)
sns.histplot(untrained_bs, color="skyblue", label="Untrained CTDS Proposal", kde=False, stat="density", binrange=(0.2, 1.0), bins=20, alpha=0.5)
sns.histplot(trained_bs, color="orange", label="Trained Proposal", kde=False, stat="density", binrange=(0.2, 1.0), bins=20, alpha=0.5)

# Format x-axis and layout
plt.xlim(0.2, 1.0)
plt.xlabel("Inverse Temperature", fontsize=12)
plt.ylabel("Density", fontsize=12)
plt.legend(prop={'size': 12})
plt.title("Inverse Temperature of CTDS Proposal Samples at T=1.0")
plt.show()

### Generating Stastics
For each of the models we care about, we generate statistics for `ode_w2`, `ode_rss`, `ode_elbo`, and `ode_eubo`.

In [ ]:
import numpy as np
from tqdm import tqdm 

SAMPLES_PER_TRIAL = 2500
NUM_TRIALS = 10
NTS = 250

def compute_stats(pinn_module):
    """
    Compute the mean and variance of the proposal samples.
    """
    ode_proposal = pinn_module.build_proposal("ode")
    stats_dict = {}
    pbar = tqdm(range(NUM_TRIALS), desc="Computing statistics")
    for _ in pbar:
        ts = torch.linspace(0, 1, NTS).to(device).view(1, -1, 1).expand(SAMPLES_PER_TRIAL, -1, -1)
        metrics = ode_proposal.get_metrics(ts, label="ode")
        for key, value in metrics.items():
            if key not in stats_dict:
                stats_dict[key] = []
            stats_dict[key].append(value)
    
    stats_dict = {
        key: (np.mean(value), np.std(value)) for key, value in stats_dict.items()
    }

    # Print the statistics
    for key, (mean, std) in stats_dict.items():
        print(f"{key}: mean={mean:.2f}, std={std:.2f}")

# Overdamped w/o Jarzynski
overdamped_nets = NETSModule.load_from_checkpoint('/afs/csail.mit.edu/u/e/erives/data/research/checkpoints/fab_experiment/fab_overdamped_test_6361835/epoch=364-val_w2=25.00.ckpt').move_to_device(device)
print("\nOverdamped w/o Jarzynski")
compute_stats(
    pinn_module = overdamped_nets,
)
# Overdamped w/ Jarzynski
overdamped_nets_jarzynski = NETSModule.load_from_checkpoint('/afs/csail.mit.edu/u/e/erives/data/research/checkpoints/fab_experiment/fab_overdamped_test_6361835/epoch=364-val_w2=25.00.ckpt').move_to_device(device)
print("\nOverdamped w/ Jarzynski")
compute_stats(
    pinn_module = overdamped_nets_jarzynski,
)

# Inertial w/o Jarzynski
inertial_nets = NETSModule.load_from_checkpoint('/afs/csail.mit.edu/u/e/erives/data/research/checkpoints/fab_experiment/fab_overdamped_test_6361835/epoch=364-val_w2=25.00.ckpt').move_to_device(device)
print("\nInertial w/o Jarzynski")
compute_stats(
    pinn_module = inertial_nets,
)

# Inertial w/ Jarzynski
inertial_nets_jarzynski = NETSModule.load_from_checkpoint('/afs/csail.mit.edu/u/e/erives/data/research/checkpoints/fab_experiment/fab_overdamped_test_6361835/epoch=364-val_w2=25.00.ckpt').move_to_device(device)
print("\nInertial w/ Jarzynski")
compute_stats(
    pinn_module = inertial_nets_jarzynski,
)

# CTDS w/o Jarzynski
ctds = CTDSModule.load_from_checkpoint('/afs/csail.mit.edu/u/e/erives/research/checkpoints/fab_experiment/fab_tempered_condensation_4558033/epoch=484-val_w2=18.27.ckpt').move_to_device(device)
print("\nCTDS w/o Jarzynski")
compute_stats(
    pinn_module = ctds
)

# CTDS w/ Jarzynski
ctds_jarzynski = CTDSModule.load_from_checkpoint('/afs/csail.mit.edu/u/e/erives/research/checkpoints/fab_experiment/fab_tempered_condensation_4558033/epoch=484-val_w2=18.27.ckpt').move_to_device(device)
print("\nCTDS w/ Jarzynski")
compute_stats(
    pinn_module = ctds_jarzynski
)